In [1]:
import json
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np

date = datetime.now().strftime("%Y-%m-%d")

verbose = 0
data_set = "gmtkn-cc-pVDZ"

data_path_list = sorted(
    list(Path("../validate").glob(f"*gmtkn*.csv")),
    key=lambda p: p.stat().st_ctime,
)

basis_args = "cc-pVDZ"
print(basis_args)

with open(f"new_dataset/{data_set}.json") as f:
    json_data = json.load(f)

with open(f"./subset.json") as f:
    full_subset_dict = json.load(f)["full_subset_dict"]
    # full_subset_dict = json.load(f)["full_subset_dict_test"]

data_subset = {}

for name_set, subset_list_ in full_subset_dict.items():
    for data_path in data_path_list:
        data = pd.read_csv(data_path)
        data_name = (data["name"].str.split(f"_{basis_args}").str[0]).to_numpy()
        data_dft = data["dft_ene"].to_numpy() * 627.5094733748099
        data_scf = data["scf_ene"].to_numpy() * 627.5094733748099
        data_cc = data["cc_ene"].to_numpy() * 627.5094733748099
        if "delta_d3bj" in data.columns:
            data_d3bj = data["delta_d3bj"].to_numpy() * 627.5094733748099
        else:
            data_d3bj = np.zeros_like(data_dft)
        if "delta_d3zero" in data.columns:
            data_d3zero = data["delta_d3zero"].to_numpy() * 627.5094733748099
        else:
            data_d3zero = np.zeros_like(data_dft)

        if modified_dft_d3bj := data.get("modified_dft_d3bj"):
            data_dft_d3bj = modified_dft_d3bj.to_numpy()
        else:
            data_dft_d3bj = data_d3bj
        if modified_dft_d3zero := data.get("modified_dft_d3zero"):
            data_dft_d3zero = modified_dft_d3zero.to_numpy()
        else:
            data_dft_d3zero = data_d3zero

        if modified_ai_d3bj := data.get("modified_ai_d3bj"):
            data_ai_d3bj = modified_ai_d3bj.to_numpy()
        else:
            data_ai_d3bj = data_d3bj
        if modified_ai_d3zero := data.get("modified_ai_d3zero"):
            data_ai_d3zero = modified_ai_d3zero.to_numpy()
        else:
            data_ai_d3zero = data_d3zero

        del data_d3bj, data_d3zero

        for i_subset in subset_list_:
            data_path_name = data_path.stem.split("_atom-1-")[1].split("_gmtkn")[0]
            name_subset = f"{data_path_name}_{i_subset}"
            data_subset[name_subset] = {
                "name": [],
                "dft": [],
                "dft_d3bj": [],
                "dft_d3zero": [],
                "ai": [],
                "ai_d3bj": [],
                "ai_d3zero": [],
                "cc": [],
            }

            reaction_dict = json_data[f"reaction-{i_subset}"]
            for i_reaction_name, i_reaction in reaction_dict.items():
                systems_list = i_reaction["systems"]
                stoichiometry_list = i_reaction["stoichiometry"]

                atomic_energy_dft = 0
                atomic_energy_dft_d3bj = 0
                atomic_energy_dft_d3zero = 0
                atomic_energy_ai = 0
                atomic_energy_ai_d3bj = 0
                atomic_energy_ai_d3zero = 0
                atomic_energy_cc = 0
                finished, exist = True, True

                for i in range(len(systems_list)):
                    mole_name = (
                        systems_list[i]
                        if i_subset == "BH76RC"
                        else f"{i_subset}-{systems_list[i]}"
                    )
                    stoichiometry = int(stoichiometry_list[i])

                    if mole_name in json_data:
                        if isinstance(json_data[mole_name], str):
                            mole_name = json_data[mole_name]
                    else:
                        finished, exist = False, False
                        if verbose > 0:
                            print(f"Warning: {mole_name} not found in data json, ERROR")
                        break

                    col = np.where(data_name == mole_name)[0]
                    if col.size == 1:
                        atomic_energy_dft += data_dft[col[0]] * stoichiometry
                        atomic_energy_dft_d3bj += data_dft_d3bj[col[0]] * stoichiometry
                        atomic_energy_dft_d3zero += (
                            data_dft_d3zero[col[0]] * stoichiometry
                        )
                        atomic_energy_ai += data_scf[col[0]] * stoichiometry
                        atomic_energy_ai_d3bj += data_ai_d3bj[col[0]] * stoichiometry
                        atomic_energy_ai_d3zero += (
                            data_ai_d3zero[col[0]] * stoichiometry
                        )
                        atomic_energy_cc += data_cc[col[0]] * stoichiometry
                    else:
                        finished = False
                        if verbose > 0:
                            print(
                                f"Warning: {mole_name} not found in {data_path.stem} data file"
                            )
                        break

                if exist:
                    data_subset[name_subset]["name"].append(i_reaction_name)
                if finished:
                    data_subset[name_subset]["dft"].append(
                        abs(atomic_energy_dft - atomic_energy_cc)
                    )
                    data_subset[name_subset]["dft_d3bj"].append(
                        abs(
                            atomic_energy_dft
                            + atomic_energy_dft_d3bj
                            - atomic_energy_cc
                        )
                    )
                    data_subset[name_subset]["dft_d3zero"].append(
                        abs(
                            atomic_energy_dft
                            + atomic_energy_dft_d3zero
                            - atomic_energy_cc
                        )
                    )
                    data_subset[name_subset]["ai"].append(
                        abs(atomic_energy_ai - atomic_energy_cc)
                    )
                    data_subset[name_subset]["ai_d3bj"].append(
                        abs(atomic_energy_ai + atomic_energy_ai_d3bj - atomic_energy_cc)
                    )
                    data_subset[name_subset]["ai_d3zero"].append(
                        abs(
                            atomic_energy_ai
                            + atomic_energy_ai_d3zero
                            - atomic_energy_cc
                        )
                    )
                    data_subset[name_subset]["cc"].append(abs(atomic_energy_cc))
                    if np.abs(atomic_energy_cc) > 1000:
                        print(
                            f"Warning: {i_reaction_name} in {name_subset} has a large CC energy: {atomic_energy_cc} kcal/mol"
                        )

            for key, val in data_subset.items():
                for key2, val2 in val.items():
                    if isinstance(val2, list):
                        data_subset[key][key2] = np.array(val2)

data_path_name_list = [
    data_path.stem.split("_atom-1-")[1].split("_gmtkn")[0]
    for data_path in data_path_list
]
header = pd.MultiIndex.from_product(
    [
        data_path_name_list,
        [
            "AI",
            "DFT",
            "AI_D3BJ",
            "DFT_D3BJ",
            "AI_D3ZERO",
            "DFT_D3ZERO",
            "Processed",
        ],
    ],
    names=["data_path", "Disp type"],
)

df_summary_subset = pd.DataFrame(columns=header)
mean_subset = pd.DataFrame(columns=header)
wtmad_1_subset = pd.DataFrame(columns=header)
wtmad_2_subset = pd.DataFrame(columns=header)
for data_path in data_path_list:
    mean_absolute_deviation_list = []
    data_path_name = data_path.stem.split("_atom-1-")[1].split("_gmtkn")[0]
    for name_set, subset_list_ in full_subset_dict.items():
        subset_ai = {}
        wtmad_1_ai = {}
        wtmad_2_ai = {}
        subset_dft = {}
        wtmad_1_dft = {}
        wtmad_2_dft = {}
        for d3_name in ["", "_d3bj", "_d3zero"]:
            subset_ai[d3_name] = []
            wtmad_1_ai[d3_name] = []
            wtmad_2_ai[d3_name] = []
            subset_dft[d3_name] = []
            wtmad_1_dft[d3_name] = []
            wtmad_2_dft[d3_name] = []
        processed = []
        for i_subset in subset_list_:
            name_subset = f"{data_path_name}_{i_subset}"
            if len(data_subset[name_subset]["ai"]) == 0:
                for col_name in [
                    "AI",
                    "DFT",
                    "AI_D3BJ",
                    "DFT_D3BJ",
                    "AI_D3ZERO",
                    "DFT_D3ZERO",
                ]:
                    df_summary_subset.loc[i_subset, (data_path_name, col_name)] = 0
                df_summary_subset.loc[i_subset, (data_path_name, "Processed")] = (
                    f"0 / {len(data_subset[name_subset]['name'])}"
                )
            else:
                for d3_name in ["", "_d3bj", "_d3zero"]:
                    df_summary_subset.loc[
                        i_subset, (data_path_name, f"AI{d3_name.upper()}")
                    ] = np.mean(data_subset[name_subset][f"ai{d3_name}"])
                    df_summary_subset.loc[
                        i_subset, (data_path_name, f"DFT{d3_name.upper()}")
                    ] = np.mean(data_subset[name_subset][f"dft{d3_name}"])
                df_summary_subset.loc[i_subset, (data_path_name, "Processed")] = (
                    f"{len(data_subset[name_subset]['ai'])} / "
                    f"{len(data_subset[name_subset]['name'])}"
                )

                if np.mean(data_subset[name_subset]["cc"]) > 75:
                    wtmad_1 = 0.1
                elif np.mean(data_subset[name_subset]["cc"]) < 7.5:
                    wtmad_1 = 10
                else:
                    wtmad_1 = 1

                for d3_name in ["", "_d3bj", "_d3zero"]:
                    subset_ai[d3_name] = np.append(
                        subset_ai[d3_name], data_subset[name_subset][f"ai{d3_name}"]
                    )
                    subset_dft[d3_name] = np.append(
                        subset_dft[d3_name], data_subset[name_subset][f"dft{d3_name}"]
                    )
                    wtmad_1_ai[d3_name] = np.append(
                        wtmad_1_ai[d3_name],
                        wtmad_1 * np.mean(data_subset[name_subset][f"ai{d3_name}"]),
                    )
                    wtmad_1_dft[d3_name] = np.append(
                        wtmad_1_dft[d3_name],
                        wtmad_1 * np.mean(data_subset[name_subset][f"dft{d3_name}"]),
                    )
                    wtmad_2_ai[d3_name] = np.append(
                        wtmad_2_ai[d3_name],
                        data_subset[name_subset][f"ai{d3_name}"]
                        / np.mean(data_subset[name_subset]["cc"]),
                    )
                    wtmad_2_dft[d3_name] = np.append(
                        wtmad_2_dft[d3_name],
                        data_subset[name_subset][f"dft{d3_name}"]
                        / np.mean(data_subset[name_subset]["cc"]),
                    )
                mean_absolute_deviation_list = np.append(
                    mean_absolute_deviation_list,
                    data_subset[name_subset]["cc"],
                )
            if (
                len(data_subset[name_subset]["ai"])
                == len(data_subset[name_subset]["name"])
                and len(data_subset[name_subset]["name"]) > 0
            ):
                processed.append(1)
            else:
                processed.append(0)
        for d3_name in ["", "_d3bj", "_d3zero"]:
            mean_subset.loc[name_set, (data_path_name, f"AI{d3_name.upper()}")] = (
                np.mean(subset_ai[d3_name])
            )
            mean_subset.loc[name_set, (data_path_name, f"DFT{d3_name.upper()}")] = (
                np.mean(subset_dft[d3_name])
            )
            wtmad_1_subset.loc[name_set, (data_path_name, f"AI{d3_name.upper()}")] = (
                np.mean(wtmad_1_ai[d3_name])
            )
            wtmad_1_subset.loc[name_set, (data_path_name, f"DFT{d3_name.upper()}")] = (
                np.mean(wtmad_1_dft[d3_name])
            )
            wtmad_2_subset.loc[name_set, (data_path_name, f"AI{d3_name.upper()}")] = (
                np.sum(wtmad_2_ai[d3_name])
            )
            wtmad_2_subset.loc[name_set, (data_path_name, f"DFT{d3_name.upper()}")] = (
                np.sum(wtmad_2_dft[d3_name])
            )
        mean_subset.loc[name_set, (data_path_name, "Processed")] = (
            f"{sum(processed)} / " f"{len(processed)}"
        )
        wtmad_1_subset.loc[name_set, (data_path_name, "Processed")] = (
            f"{sum(processed)} / " f"{len(processed)}"
        )
        wtmad_2_subset.loc[name_set, (data_path_name, "Processed")] = (
            f"{sum(processed)} / " f"{len(processed)}"
        )

    mean_absolute_deviation = np.mean(mean_absolute_deviation_list) / len(mean_absolute_deviation_list)
    for name_set in full_subset_dict.keys():
        for d3_name in ["", "_d3bj", "_d3zero"]:
            wtmad_2_subset.loc[name_set, (data_path_name, f"AI{d3_name.upper()}")] = (
                mean_absolute_deviation
                * wtmad_2_subset.loc[name_set, (data_path_name, f"AI{d3_name.upper()}")]
            )
            wtmad_2_subset.loc[name_set, (data_path_name, f"DFT{d3_name.upper()}")] = (
                mean_absolute_deviation
                * wtmad_2_subset.loc[
                    name_set, (data_path_name, f"DFT{d3_name.upper()}")
                ]
            )

print("Summary")
print("MAE")
display(mean_subset)
print("wtmad_1")
display(wtmad_1_subset)
print("wtmad_2")
display(wtmad_2_subset)
print("Summary of Subset")
print("MAE")
display(df_summary_subset)

# save summary to csv with date
df_summary_subset.to_csv(f"../validate/summary_subset_{date}.csv")
# save summary to excel with date
df_summary_subset.to_excel(f"../validate/summary_subset_{date}.xlsx")

cc-pVDZ
Summary
MAE


/home/dhem/anaconda3/envs/pyscf/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/dhem/anaconda3/envs/pyscf/lib/python3.12/site-packages/numpy/_core/_methods.py:145: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/home/dhem/anaconda3/envs/pyscf/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/dhem/anaconda3/envs/pyscf/lib/python3.12/site-packages/numpy/_core/_methods.py:145: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/home/dhem/anaconda3/envs/pyscf/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/dhem/anaconda3/envs/pyscf/lib/python3.12/site-packages/numpy/_core/_methods.

data_path        159476                                                    \
Disp type            AI        DFT       AI_D3BJ   DFT_D3BJ     AI_D3ZERO   
sub1        6425.472972  13.710857   6425.406121  14.259701   6425.316868   
sub2        2651.036347   6.612164   2659.737817   9.130035   2655.999316   
sub3       14716.588105   6.647378  14717.086971   7.856168  14716.895781   
sub4         234.418861    0.72616     234.90058   1.150397    234.959588   
sub5                NaN        NaN           NaN        NaN           NaN   

data_path                        1424849                       ...            \
Disp type DFT_D3ZERO Processed        AI        DFT   AI_D3BJ  ... AI_D3ZERO   
sub1       13.713797   18 / 18  2.002477  13.710857  2.341162  ...  1.840461   
sub2        6.757695     7 / 9  5.510109   6.612164   9.25764  ...  6.192243   
sub3        7.422183     5 / 7  2.617318   6.261376  3.548881  ...  3.228945   
sub4        1.219187    1 / 12   1.93524   3.272197  2.924356  ...  2.900361   
sub5             NaN     0 / 9  1.626727   1.321288   1.52729  ...  1.545785   

data_path                        1477959                                  \
Disp type DFT_D3ZERO Processed        AI        DFT   AI_D3BJ   DFT_D3BJ   
sub1       13.713797   18 / 18  4.024911  13.710857  4.394827  14.259701   
sub2        6.757695     7 / 9  6.165144   6.612164  9.243087   9.130035   
sub3        6.934991     7 / 7  3.280836   6.261376   4.21537   7.355579   
sub4        4.988172   11 / 12  1.861269   3.272197  2.940841    5.03074   
sub5        0.872793     8 / 9  2.264135   1.321288   1.52166   0.928323   

data_path                                 
Disp type AI_D3ZERO DFT_D3ZERO Processed  
sub1       3.888615  13.713797   18 / 18  
sub2        6.56423   6.757695     7 / 9  
sub3        3.92612   6.934991     7 / 7  
sub4       2.901616   4.988172   11 / 12  
sub5       1.487843   0.872793     8 / 9  

[5 rows x 21 columns]

wtmad_1


data_path        159476                                                    \
Disp type            AI        DFT       AI_D3BJ   DFT_D3BJ     AI_D3ZERO   
sub1       10407.833418   7.420922   10408.12846   7.414578  10407.692607   
sub2       16760.007227   12.53052  16762.220946   9.968131  16761.278604   
sub3       35835.303131   7.071498  35835.988784   8.256972  35835.615629   
sub4        1362.667514  11.609785   1360.866701  13.972834   1361.635309   
sub5                NaN        NaN           NaN        NaN           NaN   

data_path                         1424849                        ...  \
Disp type DFT_D3ZERO Processed         AI        DFT    AI_D3BJ  ...   
sub1        7.102094   18 / 18   3.239113   7.420922   2.966668  ...   
sub2       10.005264     7 / 9  13.142366   12.53052    11.9277  ...   
sub3        7.680624     5 / 7   4.832263   6.984991   5.854163  ...   
sub4       14.994136    1 / 12  11.556831  11.029773  13.901938  ...   
sub5             NaN     0 / 9  15.335367  11.571826  14.145431  ...   

data_path                                    1477959                        \
Disp type  AI_D3ZERO DFT_D3ZERO Processed         AI        DFT    AI_D3BJ   
sub1        2.651677   7.102094   18 / 18   3.789626   7.420922   3.443295   
sub2       11.524608  10.005264     7 / 9  11.038943   12.53052   9.121917   
sub3        5.522858   7.521852     7 / 7   5.389792   6.984991   6.327282   
sub4       12.876599  13.798614   11 / 12  11.131577  11.029773  14.344735   
sub5       14.544885   7.523212     8 / 9  20.741529  11.571826    13.6042   

data_path                                            
Disp type  DFT_D3BJ  AI_D3ZERO DFT_D3ZERO Processed  
sub1       7.414578   3.046364   7.102094   18 / 18  
sub2       9.968131   9.389425  10.005264     7 / 9  
sub3       8.150912   6.104995   7.521852     7 / 7  
sub4       14.87552  13.234994  13.798614   11 / 12  
sub5       8.092645  13.667041   7.523212     8 / 9  

[5 rows x 21 columns]

wtmad_2


data_path        159476                                                  \
Disp type            AI       DFT       AI_D3BJ  DFT_D3BJ     AI_D3ZERO   
sub1        8985.777191  9.060549   8985.698476  9.219193      8985.427   
sub2        8570.595373  8.151589   8573.306851  5.327781   8572.344654   
sub3       19458.738307  5.637103  19459.109746  6.616453  19458.967473   
sub4        2916.779879  2.582984   2921.208121  5.619689   2921.654373   
sub5                0.0       0.0           0.0       0.0           0.0   

data_path                        1424849                      ...            \
Disp type DFT_D3ZERO Processed        AI       DFT   AI_D3BJ  ... AI_D3ZERO   
sub1        8.946353   18 / 18  1.438234  4.040696  1.379994  ...  1.263846   
sub2         5.70819     7 / 9  3.339073  3.635331  2.970771  ...  2.735087   
sub3        6.274831     5 / 7  1.303233  2.611064   1.66299  ...  1.549133   
sub4        6.091645    1 / 12  4.683102  4.159493  6.811273  ...  6.578153   
sub5             0.0     0 / 9  6.331811   4.66236   6.30221  ...  6.387196   

data_path                        1477959                                \
Disp type DFT_D3ZERO Processed        AI       DFT   AI_D3BJ  DFT_D3BJ   
sub1        3.989769   18 / 18  1.921023  4.040696  1.839362  4.111446   
sub2        2.545658     7 / 9  3.675662  3.635331  2.522968  2.376009   
sub3        2.873647     7 / 7   1.54398  2.611064  1.925105  3.040013   
sub4         6.62018   11 / 12  4.306403  4.159493  6.554155  6.856534   
sub5         3.19725     8 / 9  8.384364   4.66236  5.942533  3.511844   

data_path                                 
Disp type AI_D3ZERO DFT_D3ZERO Processed  
sub1        1.68376   3.989769   18 / 18  
sub2       2.765968   2.545658     7 / 9  
sub3       1.817199   2.873647     7 / 7  
sub4       6.318974    6.62018   11 / 12  
sub5        5.89853    3.19725     8 / 9  

[5 rows x 21 columns]

Summary of Subset
MAE


data_path        159476                                                    \
Disp type            AI        DFT       AI_D3BJ   DFT_D3BJ     AI_D3ZERO   
W4_11        317.945892  29.506863    317.751908  31.211753    317.811085   
G21EA        140.247439   9.754522    140.249763   9.751745    140.245951   
G21IP        122.428215   8.953334    122.420929   8.946699    122.428451   
DIPCS10      653.436874  12.310852     653.46012  12.334099    653.390676   
PA26          53.123666   2.200766     52.780721   1.900152     52.837378   
SIE4x4        30.092303  21.908516     29.663006  22.337814     29.660892   
ALKBDE10     603.643712  18.125246    604.109355  18.838948    603.653407   
YBDE18       604.829954   8.145393    607.382289   7.265227    605.922782   
AL2X6         43.909343   5.659145     46.895518   1.332003       44.9166   
HEAVYSB11   43994.56611   5.391476  43993.472993   8.010474  43993.530893   
NBPRC       43867.34502    2.23255  43868.658819   2.544088  43868.688191   
ALK8       43839.655947   4.400063  43845.336779   3.174803  43841.787588   
RC21          59.650852   4.820926     60.547198   6.671006     60.051223   
G2RC       27293.139428   5.917203  27293.577909   6.933515   27293.22896   
BH76RC     13117.627303   3.484561  13117.736413   3.539828  13117.743508   
FH51       12992.795787   3.703657  12991.397263    3.39478  12991.548282   
TAUT15        29.733018    2.16453     29.660139   2.175485     29.420529   
DC13         341.474691  13.090861    336.042219  12.474047    338.636592   
MB16_43     1324.174853  15.461604   1358.878023  36.862998   1343.104292   
DARC          21.028751   10.75415     28.749681    3.43411     26.204931   
RSE43      11519.580867     3.1576  11519.615995   2.916276  11519.682847   
BSR36          0.962172    8.40118      7.660361   1.213694      5.628912   
CDIE20        18.347455   1.599507     18.123086   1.336544     18.123061   
ISO34          5.232164   2.001743      4.914145   1.577125      4.954521   
ISOL24                0          0             0          0             0   
C60ISO                0          0             0          0             0   
PArel        178.112679   1.743933    178.094383   1.733749    177.979235   
BH76       27523.294223   9.107946   27523.21364     9.8892  27523.262202   
BHPERI        39.055212   3.268992     41.915204   7.804724       41.0607   
BHDIV10      120.313041   6.277706    121.505876   7.448955     120.77316   
INV24         85.397656   2.953015     85.133394   2.656199     85.232977   
BHROT27    21864.052116   0.888365  21864.068668   0.902173  21864.016968   
PX13           7.889305  11.042023      8.673293   11.82601      8.067139   
WCPT18      4430.651319   7.967151   4430.793408   9.151986   4430.743546   
RG18         400.460499   0.230018    401.162913   0.655714    401.216751   
ADIM6          3.272419   2.181125      0.577669   1.593155      0.723773   
S22            5.067335   1.071792      6.519427   1.942981      6.550068   
S66                   0          0             0          0             0   
HEAVY28               0          0             0          0             0   
WATER27               0          0             0          0             0   
CARBHB12              0          0             0          0             0   
PNICO23               0          0             0          0             0   
HAL59                 0          0             0          0             0   
AHB21                 0          0             0          0             0   
CHB6                  0          0             0          0             0   
IL16                  0          0             0          0             0   
IDISP                 0          0             0          0             0   
ICONF                 0          0             0          0             0   
ACONF                 0          0             0          0             0   
Amino20x4             0          0             0          0             0

In [4]:
# import numpy as np
np.max(mean_absolute_deviation_list), np.sum(mean_absolute_deviation_list), np.mean(
    mean_absolute_deviation_list
)

(np.float64(1207.482739432191),
 np.float64(98414.09024557225),
 np.float64(71.1084467092285))

In [5]:
mole_name

'BH76RC-NH'

In [9]:
e1 = -4.6025270679642568e02  # orca fc DLPNO-CCSD
e2 = -4.6025221729046751e02  # orca fc CCSD
e3 = -4.6025767731174767e02 # orca nfc CCSD
e4 = -4.6025796979352134e02 # orca nfc DLPNO-CCSD
e = -460.257677434825  # pyscf CCSD
print((np.array([e1, e2, e3, e4]) - e) * 627.5094733748099)
print((np.array([e1, e2, e3, e4]) - e) / e)

[ 3.11912268e+00  3.42629231e+00  7.72321803e-05 -1.83457852e-01]
[-1.07996860e-05 -1.18632336e-05 -2.67409583e-10  6.35206561e-07]


In [14]:
e = -460.2602125077868 # pyscf CCSD(t)
e1 = -4.6026021233042377e02 # orca nfc CCSD(t)
e2 = -4.6026021233042360e02 # orca old nfc CCSD(t)
e3 = -4.6026042971888393e02  # orca nfc DLPNO-CCSD(t)
e4 = -4.6025500524617820e02
print((np.array([e1, e2, e3, e4]) - e) * 627.5094733748099)
print((np.array([e1, e2, e3, e4]) - e) / e)

[ 1.11296967e-04  1.11297074e-04 -1.36302021e-01  3.26760599e+00]
[-3.85353766e-10 -3.85354136e-10  4.71931076e-07 -1.13137340e-05]


In [ ]:
np.array([[-4.366423914512e-02], [0.000000000000e00], [-5.569827800646e-01]])
np.array([[-4.366423911029e-02], [0.000000000000e00], [-5.569827799431e-01]])

[[-4.366423908914e-02], [0.000000000000e00], [-5.569827798789e-01]]

array([[-3.48300000e-11],
       [ 0.00000000e+00],
       [-1.21500032e-10]])